# DeepSeek-R1 時系列予測ファインチューニング実験ノートブック

このノートブックでは、時系列データの前処理からファインチューニング、評価までの一連のプロセスを実験します。

## 1. セットアップ

In [ ]:
import sys
from pathlib import Path

# プロジェクトルートをパスに追加
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import TimeSeriesPreprocessor
from src.dataset_builder import TimeSeriesDatasetBuilder

# プロット設定
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("セットアップ完了")

## 2. サンプルデータの生成と確認

In [ ]:
# サンプル時系列データの読み込み
data_path = project_root / "data" / "sample_timeseries.csv"
df = pd.read_csv(data_path)

print(f"データ形状: {df.shape}")
print(f"\n最初の5行:")
display(df.head())

print(f"\n基本統計:")
display(df.describe())

In [ ]:
# データの可視化
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['value'], linewidth=1.5)
plt.xlabel('インデックス')
plt.ylabel('値')
plt.title('時系列データの可視化')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. データ前処理

In [ ]:
# 前処理器の初期化
preprocessor = TimeSeriesPreprocessor(scaling_method="minmax")

# データのクリーニング
df_clean = preprocessor.clean_data(df.copy())

# 特徴量作成
df_features = preprocessor.create_features(df_clean, timestamp_col="timestamp")

# スケーリング
df_scaled, scaler_info = preprocessor.scale_data(df_features, value_columns=["value"])

print("前処理完了")
display(df_scaled.head())

## 4. LLM用データセット構築

In [ ]:
# データセットビルダーの初期化
builder = TimeSeriesDatasetBuilder(
    sequence_length=10,
    prediction_horizon=1,
    prompt_template="default"
)

# データセット構築
dataset = builder.build_dataset_from_dataframe(
    df_scaled,
    timestamp_col="timestamp",
    value_col="value"
)

print(f"データセット作成完了: {len(dataset)} サンプル")
print(f"\nサンプル例:")
print("\nInstruction:")
print(dataset[0]['instruction'])
print("\nOutput:")
print(dataset[0]['output'])

In [ ]:
# 訓練/検証分割
train_data, val_data = builder.split_dataset(dataset, train_ratio=0.8)

print(f"訓練データ: {len(train_data)} サンプル")
print(f"検証データ: {len(val_data)} サンプル")

## 5. プロンプトテンプレートの比較実験

In [ ]:
# 異なるプロンプトテンプレートで同じデータを変換
templates = ["default", "conversational", "numeric_only"]

for template in templates:
    builder_temp = TimeSeriesDatasetBuilder(
        sequence_length=5,
        prediction_horizon=1,
        prompt_template=template
    )
    
    sample_data = df_scaled.iloc[:6]
    timestamps = sample_data["timestamp"].astype(str).tolist()
    values = sample_data["value"].tolist()
    
    prompt = builder_temp.create_prompt(timestamps[:5], values[:5])
    response = builder_temp.create_response([timestamps[5]], [values[5]])
    
    print(f"\n{'='*60}")
    print(f"テンプレート: {template}")
    print(f"{'='*60}")
    print("\nプロンプト:")
    print(prompt)
    print("\n応答:")
    print(response)

## 6. ハイパーパラメータの分析

In [ ]:
# 異なるシーケンス長でのデータセットサイズ比較
sequence_lengths = [5, 10, 20, 30]
dataset_sizes = []

for seq_len in sequence_lengths:
    builder_temp = TimeSeriesDatasetBuilder(
        sequence_length=seq_len,
        prediction_horizon=1,
        prompt_template="default"
    )
    dataset_temp = builder_temp.build_dataset_from_dataframe(
        df_scaled,
        timestamp_col="timestamp",
        value_col="value"
    )
    dataset_sizes.append(len(dataset_temp))

# プロット
plt.figure(figsize=(10, 5))
plt.bar(sequence_lengths, dataset_sizes, color='steelblue', alpha=0.7)
plt.xlabel('シーケンス長')
plt.ylabel('データセットサイズ')
plt.title('シーケンス長とデータセットサイズの関係')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("シーケンス長とデータセットサイズ:")
for seq_len, size in zip(sequence_lengths, dataset_sizes):
    print(f"  シーケンス長 {seq_len}: {size} サンプル")

## 7. データセットの保存

In [ ]:
# データセットをJSON形式で保存
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

builder.save_dataset(train_data, output_dir / "train.json", format="json")
builder.save_dataset(val_data, output_dir / "val.json", format="json")

print(f"データセットを保存しました: {output_dir}")

## 8. まとめ

この実験ノートブックでは以下を実施しました:

1. サンプルデータの読み込みと確認
2. データの前処理（クリーニング、特徴量作成、スケーリング）
3. LLM用データセットの構築
4. 異なるプロンプトテンプレートの比較
5. シーケンス長とデータセットサイズの関係分析
6. データセットの保存

次のステップ:
- `scripts/finetune.py`を使用してモデルをファインチューニング
- `src/inference.py`で予測を実行
- `scripts/evaluate.py`でモデル性能を評価